In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=KaUdtA5wL9gKln49skHD8VCIpu86tP&access_type=offline&code_challenge=d6L8obW9gbRIqRU0WWg7yv6103NMrX3VGg894Pdr21c&code_challenge_method=S256


Credentials saved to file: [C:\Users\L137844\AppData\Roaming\gcloud\application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "dev-mq-tech-transfer" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


In [6]:
import google.auth

_, project_id = google.auth.default()
print(f"Project ID: {project_id}")

Project ID: dev-mq-tech-transfer


In [7]:
# Cell 1 — instrumentation (run once per kernel session)
from phoenix.otel import register
from openinference.instrumentation.google_adk import GoogleADKInstrumentor

tracer_provider = register(
    project_name="gap_assessment", # set your project name here when you start the phoenix server
    endpoint="http://localhost:6006/v1/traces"
)
GoogleADKInstrumentor().instrument(tracer_provider=tracer_provider)

Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


OpenTelemetry Tracing Details
|  Phoenix Project: gap_assessment
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [8]:
import json
import os
from typing import Any, Dict, List, Optional

import vertexai
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.api_core.client_options import ClientOptions
from google.cloud import logging as gcp_logging
from google.cloud import discoveryengine_v1 as discoveryengine
from google.genai import types as genai_types
from vertexai.generative_models import GenerativeModel, GenerationConfig

<!-- ## Chunk-Based RAG Workflow -->

In [9]:
# ---------------------------------------------------------------------------
# System prompt
# ---------------------------------------------------------------------------

CONVERSATIONAL_PROMPT = """\
You are a TS/MS (Technical Services/Manufacturing Science) Lead Scientist supporting a drug product technology transfer of Multiple-dose [COMPOUND_1] Injection (3 mL cartridge) from the Sending Site to a Receiving Site, for Registration Stability / Process Validation batches, as described in the technology transfer gap assessment report (LQP-230-1).

A collection of technical documents is available to you, including: Gap Assessment / Risk Assessment reports (per LQP-230-1), Global Process Flow Documents (GLO_gPFD), Technical Study Summary Reports (VPHP evaluation, semi-dried residues downtime), Qualification Strategies (visual inspection, APS), and site-specific assessments.

IMPORTANT - Tool Use:
Call search_datastore exactly once, passing the user's question as the query. The tool internally handles query expansion and retrieves all relevant passages in a single call. After receiving the results, compile your complete analysis and produce a single, final response. Do not call search_datastore multiple times.

Your Task:
- Identify all facility-related requirements documented at the Sending Site and from global standards (EU Annex 1, GQS202, LQP-230-1).
- Identify all facility-related evidence available from the Receiving Site.
- Analyse requirements and evidence, compare them, and prepare the gaps.

For each gap found, extract and present the following in a structured JSON:

{
  "facility_area": "",
  "sending_site_requirements": [{"requirement": "", "source": ""}, ...],
  "global_requirements": [{"requirement": "", "source": ""}, ...],
  "receiving_site_evidence": [{"evidence": "", "source": ""}, ...],
  "gaps": [{"gap_description": "", "risk_to_product_quality": "", "cqa_at_risk": [], "owner": "", "due_date": ""}]
}

Facility areas to cover include (but are not limited to):

- Facility design, capacity, and room classifications (including dedicated line status, aseptic design meeting EU Annex 1 and GQS202)
- Air pressurization schemes and HVAC / RABS / Isolator qualification (including Grade A isolator qualification status)
- People, material, waste, and equipment flow (including flows meeting EU Annex 1 and GQS202 requirements)
- Cross-contamination risk and product mix-up controls (particularly relevant where the Receiving Site shares building space with other product lines)
- Environmental controls (humidity, temperature - e.g., NMT 40% RH for [COMPOUND_1] dispensing and DS handling)
- Utilities: WFI (target 20 degrees C for formulation, added via spray ball), Clean Steam, Process Air, Nitrogen (not required for [COMPOUND_1])
- Dispensing areas and glove box / PTB equipment readiness (including operator training status and placebo handling tests)
- Storage, staging, and sampling areas (including DS hold times, TOR, and hold time qualification)
- Aseptic Process Simulation (APS) readiness (including local aseptic hold time definition and challenge)
- Equipment parts / consumables preparation and cleaning validation (including autoclave / washer loading patterns, clean/dirty hold times, and cleaning validation qualification status)

Ground Rules:
- Be precise. Cite the source document and section for each requirement, evidence entry, and gap.
- Do not fabricate gaps not present in the documents.
- You must populate the full JSON structure for every facility area.
- If there are no gaps for a given area, return the gaps key as an empty list [] and provide a brief explanation of why no gap exists.\
"""


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def _log_step(
    project_id: str, step: str, user_id: str, query: str, response: str, citation: Any
) -> None:
    payload = {
        "Step": step,
        "User_id": user_id,
        "User_query": query,
        "Response": response,
        "Citation": citation,
    }
    gcp_logging.Client(project=project_id).logger("rag_workflow").log_struct(
        payload, severity="INFO"
    )


def _expand_queries(query: str, n: int = 4) -> List[str]:
    """Use Vertex AI Gemini directly to generate n alternative search queries."""
    model = GenerativeModel("gemini-2.5-flash")
    prompt = (
        f"You are helping with pharmaceutical document retrieval.\n"
        f'Original question: "{query}"\n\n'
        f"Generate {n} short, specific search queries (different phrasings/aspects) "
        f"to help retrieve all relevant documents. "
        f"Output only the queries, one per line, no numbering."
    )
    resp = model.generate_content(
        prompt,
        generation_config=GenerationConfig(temperature=0.2),
    )
    return [line.strip() for line in (resp.text or "").splitlines() if line.strip()][:n]


def _chunk_search_passages(
    query: str,
    project_id: str,
    location: str,
    datastore_id: str,
    page_size: int = 8,
) -> List[Dict[str, Any]]:
    """Chunk-mode search returning passage dicts."""
    endpoint = (
        "discoveryengine.googleapis.com"
        if location.lower() == "global"
        else f"{location}-discoveryengine.googleapis.com"
    )
    serving_config = (
        f"projects/{project_id}/locations/{location}/collections/default_collection/"
        f"dataStores/{datastore_id}/servingConfigs/default_config"
    )
    spec = discoveryengine.SearchRequest.ContentSearchSpec(
        search_result_mode=discoveryengine.SearchRequest.ContentSearchSpec.SearchResultMode.CHUNKS,
    )
    req = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=page_size,
        content_search_spec=spec,
    )

    passages: List[Dict[str, Any]] = []
    client = discoveryengine.SearchServiceClient(
        client_options=ClientOptions(api_endpoint=endpoint)
    )
    for r in client.search(req).results:
        chunk = getattr(r, "chunk", None)
        if chunk is None:
            continue

        content = str(getattr(chunk, "content", "") or "").strip()
        if not content:
            continue

        doc_meta = getattr(chunk, "document_metadata", None)
        title = str(getattr(doc_meta, "title", "") or "") if doc_meta else ""
        uri = str(getattr(doc_meta, "uri", "") or "") if doc_meta else ""

        chunk_name = str(getattr(chunk, "name", "") or "")
        parts = chunk_name.split("/")
        doc_id = parts[-3] if len(parts) >= 3 else ""

        passages.append(
            {"doc_id": doc_id, "title": title, "uri": uri, "content": content}
        )
    return passages


def make_search_tool(project_id: str, location: str, datastore_id: str):
    """Return an ADK-compatible search tool with internal query expansion."""

    def search_datastore(query: str) -> str:
        """Search the document datastore for relevant chunks matching the query.

        Args:
            query: The search query to find relevant document chunks.

        Returns:
            Formatted document chunks with citation IDs, titles, URIs, and content.
        """
        sub_queries = _expand_queries(query, n=4)

        seen: set = set()
        all_passages: List[Dict[str, Any]] = []
        for sq in [query] + sub_queries:
            for p in _chunk_search_passages(sq, project_id, location, datastore_id):
                key = (p["doc_id"], p["content"][:80])
                if key not in seen:
                    seen.add(key)
                    all_passages.append(p)

        if not all_passages:
            return "No relevant documents found."

        lines = []
        for idx, p in enumerate(all_passages, 1):
            lines.append(
                f"[C{idx}]\n"
                f"title: {p['title']}\n"
                f"uri: {p['uri']}\n"
                f"content: {p['content']}"
            )
        return "\n\n".join(lines)

    return search_datastore


# ---------------------------------------------------------------------------
# Agent runner
# ---------------------------------------------------------------------------

async def run_rag_agent(
    query: str,
    user_id: str,
    project_id: str,
    location: str,
    datastore_id: str,
    model_location: str = "us-central1",
) -> str:
    vertexai.init(project=project_id, location=model_location)
    os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "true"
    os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
    os.environ["GOOGLE_CLOUD_LOCATION"] = model_location

    search_tool = make_search_tool(project_id, location, datastore_id)

    agent = Agent(
        model="gemini-2.5-flash",
        name="rag_agent",
        instruction=CONVERSATIONAL_PROMPT,
        tools=[search_tool],
    )

    session_service = InMemorySessionService()
    session = await session_service.create_session(app_name="rag_app", user_id=user_id)
    runner = Runner(agent=agent, app_name="rag_app", session_service=session_service)

    _log_step(project_id, "agent_start", user_id, query, "", [])

    result_text = ""
    user_message = genai_types.Content(
        role="user",
        parts=[genai_types.Part.from_text(text=query)],
    )
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=user_message,
    ):
        if event.is_final_response():
            if event.content and event.content.parts:
                for part in event.content.parts:
                    if part.text:
                        result_text = part.text
                        break

    _log_step(project_id, "agent_complete", user_id, query, result_text, [])

    return result_text

<!-- ## Sample Run -->


In [10]:
SAMPLE_QUERY = "tell me about the gaps in facilities"
TARGET_DATASTORE_ID = "masked-data_1781603945004"

result = await run_rag_agent(
    query=SAMPLE_QUERY,
    user_id="sample-user-001",
    project_id=project_id,
    location="us",
    datastore_id=TARGET_DATASTORE_ID,
)

print(result)

c:\Users\L137844\Downloads\gap_assesment\tredence_gap_assessment\.venv\Lib\site-packages\google\adk\tools\function_tool.py:95: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  build_function_declaration(
c:\Users\L137844\Downloads\gap_assesment\tredence_gap_assessment\.venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


```json
[
  {
    "facility_area": "Facility design, capacity, and room classifications (including dedicated line status, aseptic design meeting EU Annex 1 and GQS202)",
    "sending_site_requirements": [
      {
        "requirement": "Each manufacturing line should have a history of manufacturing similar products (3-mL semi-finished cartridges for [COMPOUND_6] product).",
        "source": "C1"
      },
      {
        "requirement": "Equipment for multiple-dose [COMPOUND_1] manufacturing at the sending site should be dedicated to [COMPOUND_1] drug product manufacturing, or appropriate controls applied for shared equipment.",
        "source": "C13"
      }
    ],
    "global_requirements": [
      {
        "requirement": "All steps in sterile product manufacturing must be completed in accordance with an appropriate Contamination Control Strategy (CCS) to prevent product contamination and control potential contamination that could adversely affect product quality.",
        "source"